In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [2]:
# Data path
ML_DIR = Path.cwd().parent
DATA_PATH = ML_DIR / 'data' / 'iris' / 'iris.csv'

In [3]:
# Model parameters
TEST_SIZE = 0.2
EPOCHS = 300
LEARNING_RATE = 0.01

In [6]:
# Load the dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (150, 6)
   id  sepal_length  sepal_width  petal_length  petal_width species
0   1           5.1          3.5           1.4          0.2  setosa
1   2           4.9          3.0           1.4          0.2  setosa
2   3           4.7          3.2           1.3          0.2  setosa
3   4           4.6          3.1           1.5          0.2  setosa
4   5           5.0          3.6           1.4          0.2  setosa


In [16]:
# Extract features
feature_columns = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X = df[feature_columns].to_numpy()

# Encode labels
label_mapping = {"setosa": 0, "versicolor": 1, "virginica": 2}
y = df["species"].map(label_mapping).values

if pd.isna(y).any():
    raise ValueError("Unknown class found in Species column.")

print(f"Class distribution: {df["species"].value_counts()}")

Class distribution: species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64


In [10]:
# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=y,
)

In [22]:
# Scale the features
# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

/tmp/ipykernel_171306/640577647.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
/tmp/ipykernel_171306/640577647.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)
/tmp/ipykernel_171306/640577647.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.long)
/tmp/ipykernel_171306/640577647.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone(

In [23]:
# Define the NN model
class IrisNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(4, 8),    # Input layer (4 features) to hidden layer (8 neurons)
            nn.ReLU(),          # RELU activation function
            nn.Linear(8, 3),    # Hidden layer (8 neurons) to output layer (3 classes)
            nn.Softmax(dim=1)   # Softmax activation for multi-class classification
        )

    def forward(self, x):
        return self.network(x)


model = IrisNet()

print(f"Model: {model}")

Model: IrisNet(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=3, bias=True)
    (3): Softmax(dim=1)
  )
)


In [24]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [25]:
# Training loop
print("\nTraining...")
for epoch in range(EPOCHS):
    model.train()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1:3d}/{EPOCHS}] "
            f"Loss: {loss.item():.4f}"
        )


Training...
Epoch [ 10/300] Loss: 1.0878
Epoch [ 20/300] Loss: 1.0295
Epoch [ 30/300] Loss: 0.9234
Epoch [ 40/300] Loss: 0.7873
Epoch [ 50/300] Loss: 0.7224
Epoch [ 60/300] Loss: 0.6858
Epoch [ 70/300] Loss: 0.6555
Epoch [ 80/300] Loss: 0.6308
Epoch [ 90/300] Loss: 0.6146
Epoch [100/300] Loss: 0.6035
Epoch [110/300] Loss: 0.5959
Epoch [120/300] Loss: 0.5910
Epoch [130/300] Loss: 0.5876
Epoch [140/300] Loss: 0.5851
Epoch [150/300] Loss: 0.5831
Epoch [160/300] Loss: 0.5816
Epoch [170/300] Loss: 0.5803
Epoch [180/300] Loss: 0.5792
Epoch [190/300] Loss: 0.5783
Epoch [200/300] Loss: 0.5774
Epoch [210/300] Loss: 0.5766
Epoch [220/300] Loss: 0.5759
Epoch [230/300] Loss: 0.5753
Epoch [240/300] Loss: 0.5747
Epoch [250/300] Loss: 0.5741
Epoch [260/300] Loss: 0.5735
Epoch [270/300] Loss: 0.5730
Epoch [280/300] Loss: 0.5725
Epoch [290/300] Loss: 0.5719
Epoch [300/300] Loss: 0.5714


In [26]:
# Evaluate the model
model.eval()

with torch.no_grad():
    outputs = model(X_test)
    predictions = torch.argmax(outputs, dim=1)

accuracy = accuracy_score(y_test.numpy(), predictions.numpy())

print(f"Test accuracy: {accuracy * 100:.4f}%")

Test accuracy: 100.0000%
